# Chapter 18 — Failure Mode Analysis with Probability
*Track 4: Engineers*

## 🎯 Learning Objectives
- Perform FMEA with probability-based Risk Priority Number (RPN)
- Build fault trees and compute top-event probability
- Apply event tree analysis for consequence modeling

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.stats as stats
import pandas as pd

rng = np.random.default_rng(42)
plt.style.use("seaborn-v0_8-whitegrid")
print("Libraries loaded ✅")

## 1. Concept Review — FMEA

**FMEA** (Failure Mode and Effects Analysis) ranks failure modes by:

$$\text{RPN} = \text{Severity} \times \text{Occurrence} \times \text{Detection}$$

Each factor is rated 1–10:
- **Severity**: how bad is the effect? (1=minor, 10=catastrophic)
- **Occurrence**: how often does it happen? (1=rare, 10=almost certain)
- **Detection**: how easy is it to detect before reaching customer? (1=easy to detect, 10=undetectable)

In [ ]:
# FMEA analysis for a pump system
fmea_data = {
    "Failure Mode": [
        "Seal leak", "Bearing failure", "Impeller wear",
        "Motor overload", "Cavitation", "Corrosion"
    ],
    "Severity":    [8, 7, 5, 9, 6, 7],
    "Occurrence":  [4, 5, 6, 3, 4, 5],
    "Detection":   [3, 4, 6, 2, 5, 7],
}
df = pd.DataFrame(fmea_data)
df["RPN"] = df["Severity"] * df["Occurrence"] * df["Detection"]
df = df.sort_values("RPN", ascending=False)
print(df.to_string(index=False))

colors = ["red" if r > 200 else "orange" if r > 100 else "green" for r in df["RPN"]]
plt.barh(df["Failure Mode"], df["RPN"], color=colors)
plt.axvline(200, color="red", linestyle="--", label="High risk (RPN>200)")
plt.axvline(100, color="orange", linestyle="--", label="Medium risk (RPN>100)")
plt.xlabel("Risk Priority Number (RPN)")
plt.title("FMEA — Pump System Risk Ranking")
plt.legend(); plt.tight_layout(); plt.show()

## 2. Math Walkthrough — Fault Tree Analysis

**AND gate**: all inputs must fail for output to fail:
$$P(\text{top}) = \prod_i P(\text{input}_i)$$

**OR gate**: any input failing causes output:
$$P(\text{top}) = 1 - \prod_i (1 - P(\text{input}_i))$$

In [ ]:
# Fault tree calculation
def and_gate(probs):
    return np.prod(probs)

def or_gate(probs):
    return 1 - np.prod(1 - np.array(probs))

# System failure tree
p_sensor_A = 0.02   # sensor A failure rate
p_sensor_B = 0.015  # sensor B
p_power_A  = 0.005
p_power_B  = 0.005

# Sensor subsystem: both must fail (AND)
p_sensor_subsystem = and_gate([p_sensor_A, p_sensor_B])

# Power subsystem: either fails (OR)
p_power_subsystem = or_gate([p_power_A, p_power_B])

# Top event: either subsystem fails (OR)
p_top = or_gate([p_sensor_subsystem, p_power_subsystem])

print(f"Sensor subsystem failure (AND): {p_sensor_subsystem:.6f}")
print(f"Power subsystem failure (OR):   {p_power_subsystem:.6f}")
print(f"Top event probability (OR):     {p_top:.6f}")
print(f"Top event per year (1e6 hours): {p_top*8760:.4f}")

## 3–6. Monte Carlo Fault Tree, Event Tree, Practice

In [ ]:
# Monte Carlo fault tree simulation
n_sim = 100_000
sensor_A_fail = rng.random(n_sim) < p_sensor_A
sensor_B_fail = rng.random(n_sim) < p_sensor_B
power_A_fail  = rng.random(n_sim) < p_power_A
power_B_fail  = rng.random(n_sim) < p_power_B

sensor_sub_fail = sensor_A_fail & sensor_B_fail
power_sub_fail  = power_A_fail | power_B_fail
top_event_fail  = sensor_sub_fail | power_sub_fail

mc_prob = top_event_fail.mean()
print(f"Monte Carlo top event probability: {mc_prob:.6f}")
print(f"Analytic:                          {p_top:.6f}")

In [ ]:
# Event tree: consequences given initiating event
# Initiating event: pressure release (IE)
p_IE = 0.01  # per year

# Safety barriers (independent)
p_detection  = 0.95  # probability detector works
p_isolation  = 0.90  # probability valve closes
p_mitigation = 0.80  # probability mitigation system works

# Consequence paths
paths = {
    "Near miss (all barriers)":      p_IE * p_detection * p_isolation * p_mitigation,
    "Minor incident (no mitigation)": p_IE * p_detection * p_isolation * (1-p_mitigation),
    "Moderate (no isolation)":        p_IE * p_detection * (1-p_isolation),
    "Major incident (no detection)":  p_IE * (1-p_detection),
}
print("Event Tree — Annual Frequencies:")
for outcome, freq in paths.items():
    print(f"  {outcome:<40}: {freq:.2e}")

plt.barh(list(paths.keys()), list(paths.values()), color=["green","yellow","orange","red"])
plt.xscale("log"); plt.xlabel("Annual frequency")
plt.title("Event Tree Analysis — Pressure Release Scenario")
plt.tight_layout(); plt.show()